In [0]:
-- * first purchase month
-- * repeat purchase rate

with cohort_month AS (
select 
 customer_unique_id as cust_id,
 date_trunc('month', min(order_purchase_timestamp)) as first_month_purchase
 from workspace.default.final_quick_comm_dataset
 group by 1
),

monthly_usage AS (
select 
customer_unique_id as cust_id,
date_trunc('month', order_purchase_timestamp) as month
from workspace.default.final_quick_comm_dataset
group by 1,2
),

user_retention AS (
    SELECT
        c.first_month_purchase,
        DATEDIFF(MONTH, c.first_month_purchase, m.month) as month_offset,
        COUNT(DISTINCT m.cust_id) as active_customers
    FROM monthly_usage m
    JOIN cohort_month c ON m.cust_id = c.cust_id
    GROUP BY 1, 2
)

SELECT
    date(first_month_purchase) as cohort_month,
    month_offset,
    active_customers,
    round(100.0 * active_customers / MAX(active_customers) OVER(PARTITION BY first_month_purchase),2) as retention_rate
FROM user_retention
where month_offset <= 12 and date(first_month_purchase) >= '2017-01-01'
ORDER BY 1, 2;

Databricks visualization. Run in Databricks to view.